In [47]:

%pip install -q pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Data preparation
#### Setup

In [48]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42

# Rutas centralizadas
TRAIN_PATH = "../ames-housing-desarrollo-estudiantes/train.csv"
VAL_PATH = "../ames-housing-desarrollo-estudiantes/validation.csv"

# Cargar datasets
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)

# Separar predictoras y target
X_train = train_df.drop(columns=["SalePrice"]).copy()
y_train_raw = train_df["SalePrice"].copy()

X_val = val_df.drop(columns=["SalePrice"]).copy()
y_val_raw = val_df["SalePrice"].copy()

print("\nX_train:", X_train.shape)
print("y_train_raw:", y_train_raw.shape)
print("X_val:", X_val.shape)
print("y_val_raw:", y_val_raw.shape)

Train: (1022, 81)
Validation: (219, 81)

X_train: (1022, 80)
y_train_raw: (1022,)
X_val: (219, 80)
y_val_raw: (219,)


#### Eliminación de features

In [49]:
X_train = X_train.drop(columns=["Id"])
X_val = X_val.drop(columns=["Id"])

if X_train["Utilities"].nunique() <= 1:
    X_train = X_train.drop(columns=["Utilities"])
    X_val = X_val.drop(columns=["Utilities"])
    
print("Cantidad de features después de las eliminaciones:", X_train.shape[1])


Cantidad de features después de las eliminaciones: 78


Por ahora, se eliminará `Id` ya que no representa ninguna característica de la vivienda

Como la variable `Utilities` solo tiene un mismo valor, se tomó la decisión de eliminarla debido a que no permite diferenciar casas y no aporta información al modelo.

#### Valores faltantes

`LotFrontage` tiene aproximadamente un 18,2% de faltantes, por lo que no conviene eliminar las filas. Además, el EDA mostró que `LotFrontage` depende bastante del contexto de la vivienda, especialmente del barrio. Por esto,  se puede utilizar la mediana por `Neighborhood`.

In [50]:
lotfrontage_medians = (
    X_train.groupby("Neighborhood")["LotFrontage"].median()
)

global_median = X_train["LotFrontage"].median()


X_train["LotFrontage"] = X_train["LotFrontage"].fillna(
    X_train["Neighborhood"].map(lotfrontage_medians)
)

X_train["LotFrontage"] = X_train["LotFrontage"].fillna(global_median)

X_val["LotFrontage"] = X_val["LotFrontage"].fillna(
    X_val["Neighborhood"].map(lotfrontage_medians)
)

X_val["LotFrontage"] = X_val["LotFrontage"].fillna(global_median)

Valores faltantes en variables categóricas que significan ausencia, por lo tanto se los imputa con `None`

In [51]:
categorical_none_features = [
    "PoolQC",
    "MiscFeature",
    "Alley",
    "Fence",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "MasVnrType"
]

for col in categorical_none_features:
    X_train[col] = X_train[col].fillna("None")
    X_val[col] = X_val[col].fillna("None")

Valores faltanes en variables numéricas que significan ausencia, por lo tanto se los imputa con `0`

In [52]:
numeric_none_features = [
    "GarageYrBlt",
    "MasVnrArea"
]

for col in numeric_none_features:
    X_train[col] = X_train[col].fillna(0)
    X_val[col] = X_val[col].fillna(0)


Como `Electrical` tiene solamente 1 valor faltante, se puede utilizar la moda. 

In [53]:
electrical_mode = X_train["Electrical"].mode()[0]

X_train["Electrical"] = X_train["Electrical"].fillna(electrical_mode)
X_val["Electrical"] = X_val["Electrical"].fillna(electrical_mode)

##### Comprobación de valores faltantes imputados correctamente

In [54]:
print("Faltantes en TRAIN:")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

print("\nFaltantes en VALIDATION:")
print(X_val.isnull().sum()[X_val.isnull().sum() > 0])

Faltantes en TRAIN:
Series([], dtype: int64)

Faltantes en VALIDATION:
Series([], dtype: int64)


#### Feature Engineering 

In [55]:
def add_features(df):
    df = df.copy()

    # Superficie total
    df["TotalSF"] = (
        df["TotalBsmtSF"] +
        df["1stFlrSF"] +
        df["2ndFlrSF"]
    )

    # Baños equivalentes
    df["TotalBathrooms"] = (
        df["FullBath"] +
        0.5 * df["HalfBath"] +
        df["BsmtFullBath"] +
        0.5 * df["BsmtHalfBath"]
    )

    # Edad de la vivienda
    df["HouseAgeAtSale"] = (
        df["YrSold"] - df["YearBuilt"]
    )

    # Edad desde remodelación
    df["RemodAgeAtSale"] = (
        df["YrSold"] - df["YearRemodAdd"]
    )

    # Remodelación
    df["IsRemodeled"] = (
        df["YearRemodAdd"] != df["YearBuilt"]
    ).astype(int)

    # Superficie de porches
    df["TotalPorchSF"] = (
        df["OpenPorchSF"] +
        df["3SsnPorch"] +
        df["EnclosedPorch"] +
        df["ScreenPorch"] +
        df["WoodDeckSF"]
    )

    # Indicadores de existencia
    df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
    df["HasBsmt"] = (df["TotalBsmtSF"] > 0).astype(int)
    df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)
    df["HasPool"] = (df["PoolArea"] > 0).astype(int)
    df["Has2ndFloor"] = (df["2ndFlrSF"] > 0).astype(int)
    df["HasMasVnr"] = (df["MasVnrArea"] > 0).astype(int)

    return df

X_train = add_features(X_train)
X_val = add_features(X_val)

#### Transformación de target

In [56]:
y_train = np.log1p(y_train_raw)
y_val = np.log1p(y_val_raw)

print("Skew original:", y_train_raw.skew())
print("Skew transformado:", y_train.skew())

Skew original: 1.9503259662422507
Skew transformado: 0.12354527821764766


#### Variables categóricas

##### Ordinales: con orden

In [57]:
ordinal_features = [
    "LotShape",
    "LandSlope",
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "HeatingQC",
    "KitchenQual",
    "FireplaceQu",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "PoolQC",
    "Functional"
]

ordinal_categories = [
    # LotShape
    ["IR3", "IR2", "IR1", "Reg"],
    # LandSlope
    ["Sev", "Mod", "Gtl"],
    # ExterQual
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # ExterCond
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # BsmtQual
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # BsmtCond
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # BsmtExposure
    ["None", "No", "Mn", "Av", "Gd"],
    # BsmtFinType1
    ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    # BsmtFinType2
    ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    # HeatingQC
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # KitchenQual
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # FireplaceQu
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # GarageFinish
    ["None", "Unf", "RFn", "Fin"],
    # GarageQual
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # GarageCond
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # PoolQC
    ["None", "Fa", "TA", "Gd", "Ex"],
    # Functional
    ["None", "Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"]
]

In [58]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder(
    categories=ordinal_categories,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

##### Nominales: sin orden

In [59]:
X_train["MSSubClass"] = X_train["MSSubClass"].astype(str)
X_val["MSSubClass"] = X_val["MSSubClass"].astype(str)

**Aclaración**: originalmente `MSSubClass` contiene enteros, pero no es realmente una variable numérica continua, sino que es un código que representa tipos de vivienda. Es por esto, que termina siendo de tipo categórica nominal.

In [60]:
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

nominal_features = [
    col for col in categorical_features
    if col not in ordinal_features
]

In [61]:
from sklearn.preprocessing import OneHotEncoder

nominal_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

#### Variables numéricas

In [62]:
skewed_features = [
    "LotArea",
    "GrLivArea",
    "1stFlrSF",
    "TotalBsmtSF",
    "BsmtFinSF1",
    "WoodDeckSF",
    "OpenPorchSF",
    "MasVnrArea",
    "BsmtUnfSF",
    "2ndFlrSF"
]

In [63]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, StandardScaler

numeric_log_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler())
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

ordinal_pipeline = Pipeline([
    ("encoder", ordinal_encoder),
    ("scaler", StandardScaler())
])

nominal_pipeline = Pipeline([
    ("encoder", nominal_encoder)
])

from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num_log", numeric_log_pipeline, numeric_log_features),
        ("num", numeric_pipeline, numeric_regular_features),
        ("ordinal", ordinal_pipeline, ordinal_features),
        ("nominal", nominal_pipeline, nominal_features)
    ],
    remainder="drop"
)

NameError: name 'numeric_log_features' is not defined

In [ ]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_val_prepared = preprocessor.transform(X_val)


print("Train original:", X_train.shape)
print("Train preparado:", X_train_prepared.shape)

print("Validation original:", X_val.shape)
print("Validation preparado:", X_val_prepared.shape)